# Inference LLM on notebooks

In [1]:
!pip -q install pyngrok

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import torch

In [5]:
from google.colab import userdata
from huggingface_hub import login

hf_token=""
login(token=hf_token)

In [6]:
NGROK_AUTH_TOKEN = ""

In [7]:
model_name = "quangne/text2diagram-AceMath-1.5B-Instruct-merged-geometry3k8-8-1-1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_cuda = torch.cuda.is_available()
load_dtype = (
    torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported())
    else (torch.float16 if use_cuda else torch.float32)
 )

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=load_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
 )
model.eval()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [12]:
import nest_asyncio
nest_asyncio.apply()

In [13]:
def prepare_inference_context(model):
    model.eval()
    use_cuda = torch.cuda.is_available() and str(model.device).startswith("cuda")
    compute_dtype = torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported()) else torch.float16
    return use_cuda, compute_dtype

In [15]:
from typing import Literal
import time
import uuid

from fastapi import FastAPI
from pydantic import BaseModel, Field
from pyngrok import ngrok
import uvicorn
import torch
import nest_asyncio

nest_asyncio.apply()
app = FastAPI()

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class ChatCompletionsRequest(BaseModel):
    model: str = "acemath-mock"
    messages: list[ChatMessage] = Field(default_factory=list)
    max_tokens: int = 256
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.08


def build_prompt(messages: list[ChatMessage]) -> str:
    # ghép messages thành 1 prompt text để feed vào model local
    lines = []
    for m in messages:
        if m.role == "system":
            lines.append(f"[System]\n{m.content}")
        elif m.role == "user":
            lines.append(f"[User]\n{m.content}")
        else:
            lines.append(f"[Assistant]\n{m.content}")
    return "\n\n".join(lines).strip()


def generate_dsl(
    prompt_text: str,
    max_new_tokens: int = 256,
    use_cuda: bool = False,
    compute_dtype: torch.dtype = torch.float16,
) -> str:
    messages = [{"role": "user", "content": prompt_text}]
    if tokenizer.chat_template:
        rendered_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        rendered_prompt = f"User:\n{prompt_text}\n\nAssistant:\n"

    inputs = tokenizer(rendered_prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        if use_cuda:
            with torch.autocast(device_type="cuda", dtype=compute_dtype):
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.08,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                    use_cache=True,
                )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.08,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
                )

    prompt_len = inputs["input_ids"].shape[-1]
    generated = outputs[0][prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


@app.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionsRequest):
    prompt_text = build_prompt(req.messages)
    if not prompt_text:
        prompt_text = ""

    use_cuda, compute_dtype = prepare_inference_context(model)
    text = generate_dsl(
        prompt_text=prompt_text,
        max_new_tokens=req.max_tokens,
        use_cuda=use_cuda,
        compute_dtype=compute_dtype,
    )

    # OpenAI-compatible response shape
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": text,
                },
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
        },
    }


ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print("PUBLIC URL:", public_url)
print("CHAT URL:", f"{public_url}/v1/chat/completions")

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000))
await server.serve()

PUBLIC URL: NgrokTunnel: "https://victoria-communicable-sometimes.ngrok-free.dev" -> "http://localhost:8000"
CHAT URL: NgrokTunnel: "https://victoria-communicable-sometimes.ngrok-free.dev" -> "http://localhost:8000"/v1/chat/completions


INFO:     Started server process [1547]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     115.73.166.151:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     115.73.166.151:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1547]
